# 5. Data Preprocessing

4번 노트북에서 만든 subject-level split과 LSTM sequence index를 사용해 모델 입력 tensor를 생성합니다. 모든 feature 제외, imputation, scaling, categorical encoding 값은 train split에서만 fit합니다.

In [ ]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd

RANDOM_STATE = 42
MISSING_THRESHOLD = 0.95

SRC_DIR = Path.cwd().resolve()
PROJECT_DIR = SRC_DIR.parent
MODELING_DIR = PROJECT_DIR / "processed" / "modeling"
PREPROCESS_DIR = PROJECT_DIR / "models" / "preprocess"
PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"MODELING_DIR: {MODELING_DIR}")
print(f"PREPROCESS_DIR: {PREPROCESS_DIR}")

In [ ]:
required_files = {
    "binned": MODELING_DIR / "events_12h_binned_with_split.csv",
    "seq_train": MODELING_DIR / "lstm_sequence_index_train.csv",
    "seq_test": MODELING_DIR / "lstm_sequence_index_test.csv",
}
missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing modeling inputs:\n" + "\n".join(missing))

binned = pd.read_csv(required_files["binned"])
seq_train = pd.read_csv(required_files["seq_train"])
seq_test = pd.read_csv(required_files["seq_test"])
seq_all = pd.concat([seq_train, seq_test], ignore_index=True)

for df_name, df in [("binned", binned), ("seq_train", seq_train), ("seq_test", seq_test)]:
    print(df_name, df.shape)

required_binned_cols = {"subject_id", "hadm_id", "stay_id", "bin", "split", "delirium"}
missing_cols = sorted(required_binned_cols - set(binned.columns))
if missing_cols:
    raise ValueError(f"binned data missing columns: {missing_cols}")

## Feature set 정의

In [ ]:
leakage_or_metadata_cols = {
    "subject_id", "hadm_id", "stay_id", "bin", "hours", "bin_start", "bin_end",
    "split", "intime", "outtime", "delirium", "ever_delirium", "los_hours", "icu_los_hours",
}

candidate_cols = [col for col in binned.columns if col not in leakage_or_metadata_cols]
numeric_cols = [col for col in candidate_cols if pd.api.types.is_numeric_dtype(binned[col])]
categorical_cols = [col for col in candidate_cols if col not in numeric_cols]

train_rows = binned["split"] == "train"
numeric_missingness = binned.loc[train_rows, numeric_cols].isna().mean().sort_values(ascending=False)
dropped_high_missing = numeric_missingness[numeric_missingness > MISSING_THRESHOLD].index.tolist()
numeric_cols = [col for col in numeric_cols if col not in dropped_high_missing]

print("categorical_cols", categorical_cols)
print("numeric_cols kept", len(numeric_cols))
print("numeric_cols dropped for high missingness", len(dropped_high_missing))
display(numeric_missingness.head(20))

## Train 기준 preprocessing fit

In [ ]:
def safe_name(value: object) -> str:
    text = str(value)
    text = re.sub(r"[^0-9A-Za-z가-힣]+", "_", text).strip("_")
    return text or "Unknown"


numeric_train = binned.loc[train_rows, numeric_cols]
numeric_medians = numeric_train.median(numeric_only=True).fillna(0.0)
numeric_means = numeric_train.fillna(numeric_medians).mean()
numeric_stds = numeric_train.fillna(numeric_medians).std(ddof=0).replace(0, 1.0).fillna(1.0)

category_levels = {}
for col in categorical_cols:
    values = binned.loc[train_rows, col].fillna("Unknown").astype(str)
    levels = sorted(values.unique().tolist())
    if "Unknown" not in levels:
        levels.append("Unknown")
    category_levels[col] = levels


def transform_binned_features(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    if numeric_cols:
        numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
        numeric = numeric.fillna(numeric_medians)
        numeric = (numeric - numeric_means) / numeric_stds
        numeric = numeric.astype("float32")
        parts.append(numeric)

    cat_parts = []
    for col in categorical_cols:
        levels = category_levels[col]
        known = set(levels)
        values = df[col].fillna("Unknown").astype(str)
        values = values.where(values.isin(known), "Unknown")
        encoded = pd.DataFrame(0.0, index=df.index, columns=[f"{col}__{safe_name(level)}" for level in levels])
        for level in levels:
            encoded.loc[values == level, f"{col}__{safe_name(level)}"] = 1.0
        cat_parts.append(encoded.astype("float32"))
    parts.extend(cat_parts)

    if not parts:
        raise ValueError("No model features were generated")
    return pd.concat(parts, axis=1)


feature_frame = transform_binned_features(binned)
feature_columns = feature_frame.columns.tolist()
feature_lookup = feature_frame.copy()
feature_lookup[["stay_id", "bin"]] = binned[["stay_id", "bin"]]
feature_lookup = feature_lookup.set_index(["stay_id", "bin"]).sort_index()

print("feature_frame", feature_frame.shape)
print("first features", feature_columns[:20])

## Sequence tensor 생성

In [ ]:
def parse_bins(text: str) -> list[int]:
    return [int(x) for x in str(text).split(",") if str(x).strip()]


def make_tensor(sequence_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    x_rows = []
    missing_keys = []
    for _, row in sequence_df.iterrows():
        stay_id = row["stay_id"]
        input_bins = parse_bins(row["input_bins"])
        step_rows = []
        for bin_id in input_bins:
            key = (stay_id, bin_id)
            if key not in feature_lookup.index:
                missing_keys.append(key)
                continue
            step_rows.append(feature_lookup.loc[key, feature_columns].to_numpy(dtype=np.float32))
        if len(step_rows) != len(input_bins):
            continue
        x_rows.append(np.stack(step_rows, axis=0))

    if missing_keys:
        raise KeyError(f"Missing {len(missing_keys)} stay/bin feature rows; first key: {missing_keys[0]}")
    x = np.stack(x_rows, axis=0).astype(np.float32)
    y_binary = sequence_df["y_any_t_to_t_plus_2"].to_numpy(dtype=np.float32)
    y_steps = sequence_df[["y_t", "y_t_plus_1", "y_t_plus_2"]].to_numpy(dtype=np.float32)
    meta_cols = ["example_id", "subject_id", "hadm_id", "stay_id", "split", "anchor_bin", "input_bins", "target_bins"]
    meta = sequence_df[meta_cols + ["y_t", "y_t_plus_1", "y_t_plus_2", "y_any_t_to_t_plus_2"]].copy()
    return x, y_binary, y_steps, meta


X_train, y_train, y_train_steps, meta_train = make_tensor(seq_train)
X_test, y_test, y_test_steps, meta_test = make_tensor(seq_test)

print("X_train", X_train.shape, "y_train", y_train.shape, "positive rate", y_train.mean())
print("X_test", X_test.shape, "y_test", y_test.shape, "positive rate", y_test.mean())
print("NaN count train/test", np.isnan(X_train).sum(), np.isnan(X_test).sum())
if np.isnan(X_train).any() or np.isnan(X_test).any():
    raise ValueError("NaNs remain in model tensors")

## 산출물 저장

In [ ]:
np.save(MODELING_DIR / "X_train_lstm.npy", X_train)
np.save(MODELING_DIR / "X_test_lstm.npy", X_test)
np.save(MODELING_DIR / "y_train_lstm.npy", y_train)
np.save(MODELING_DIR / "y_test_lstm.npy", y_test)
np.save(MODELING_DIR / "y_train_steps_lstm.npy", y_train_steps)
np.save(MODELING_DIR / "y_test_steps_lstm.npy", y_test_steps)
meta_train.to_csv(MODELING_DIR / "lstm_train_metadata.csv", index=False)
meta_test.to_csv(MODELING_DIR / "lstm_test_metadata.csv", index=False)

feature_missingness_train = binned.loc[train_rows, numeric_cols].isna().mean().rename("missing_rate").reset_index().rename(columns={"index": "feature"})
feature_missingness_test = binned.loc[binned["split"] == "test", numeric_cols].isna().mean().rename("missing_rate").reset_index().rename(columns={"index": "feature"})
feature_missingness_train.to_csv(MODELING_DIR / "feature_missingness_train.csv", index=False)
feature_missingness_test.to_csv(MODELING_DIR / "feature_missingness_test.csv", index=False)

preprocess_params = {
    "missing_threshold": MISSING_THRESHOLD,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "dropped_high_missing": dropped_high_missing,
    "feature_columns": feature_columns,
    "category_levels": category_levels,
    "target": "y_any_t_to_t_plus_2",
    "excluded_columns": sorted(leakage_or_metadata_cols),
}
with open(PREPROCESS_DIR / "lstm_feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2, ensure_ascii=False)
with open(PREPROCESS_DIR / "lstm_preprocess_params.json", "w") as f:
    json.dump(preprocess_params, f, indent=2, ensure_ascii=False)

joblib.dump(
    {
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "category_levels": category_levels,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "feature_columns": feature_columns,
    },
    PREPROCESS_DIR / "lstm_preprocessor.joblib",
)

summary = pd.DataFrame(
    [
        {"split": "train", "n_sequences": len(y_train), "n_positive": int(y_train.sum()), "positive_rate": float(y_train.mean())},
        {"split": "test", "n_sequences": len(y_test), "n_positive": int(y_test.sum()), "positive_rate": float(y_test.mean())},
    ]
)
summary.to_csv(MODELING_DIR / "lstm_preprocessing_summary.csv", index=False)
display(summary)
print("Saved preprocessing outputs")